<h2 style="color:#1f77b4;">Live Coding Parte 1</h2>

<h5 style="color:#1f77b4;">1. Importa la librería pandas</h5>

In [36]:
import pandas as pd

<h5 style="color:#1f77b4;">2. Elegir una página web con tablas HTML públicas (ej: Wikipedia)</h5>

In [37]:
url = "https://en.wikipedia.org/wiki/List_of_largest_cities"

<h5 style="color:#1f77b4;">3. Usar read_html() para extraer todas las tablas</h5>

In [38]:
#pip install requests


In [39]:
import pandas as pd
import requests

url = "https://en.wikipedia.org/wiki/List_of_largest_cities"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status() 

tablas = pd.read_html(resp.text)
print("Tablas encontradas:", len(tablas))


Tablas encontradas: 7


C:\Users\yuri1\AppData\Local\Temp\ipykernel_10688\1524441047.py:13: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tablas = pd.read_html(resp.text)


<h5 style="color:#1f77b4;">4.Seleccionar una tabla por índice</h5>

In [40]:
for i, t in enumerate(tablas[:10]):  # muestra solo las primeras 10
    print(f"Tabla {i}: {t.shape}")

Tabla 0: (10, 1)
Tabla 1: (83, 13)
Tabla 2: (6, 2)
Tabla 3: (2, 6)
Tabla 4: (1, 6)
Tabla 5: (8, 2)
Tabla 6: (9, 2)


In [41]:
df = tablas[1].copy()

In [42]:
df.columns

MultiIndex([(                         'City[a]', ...),
            (                         'Country', ...),
            ('UN 2025 population estimates[11]', ...),
            (                  'City proper[b]', ...),
            (                  'City proper[b]', ...),
            (                  'City proper[b]', ...),
            (                  'City proper[b]', ...),
            (                  'Urban area[12]', ...),
            (                  'Urban area[12]', ...),
            (                  'Urban area[12]', ...),
            (            'Metropolitan area[c]', ...),
            (            'Metropolitan area[c]', ...),
            (            'Metropolitan area[c]', ...)],
           )

In [43]:
df.head()

City[a]     Country UN 2025 population estimates[11]  \
    City[a]     Country UN 2025 population estimates[11]   
0   Jakarta   Indonesia                         41913860   
1     Dhaka  Bangladesh                         36585479   
2     Tokyo       Japan                         33412512   
3     Delhi       India                         30222405   
4  Shanghai       China                         29558908   

               City proper[b]                                         \
                   Definition Population Area (km2)   Density (/km2)   
0              Special region   10154134        664      15,292 [13]   
1                Capital city   10295407        338  30,460 [15][16]   
2       Metropolis prefecture   13515271       2191       6,169 [18]   
3  National Capital Territory   16753235       1484      11,289 [20]   
4                Municipality   24870895       6341   3,922 [22][23]   

  Urban area[12]                           Metropolitan area[c]             \
      Population Area (km2) Density (/km2)           Population Area (km2)   
0       33756000       3546      9,519 [d]             33430285       7063   
1       19134000        619          30911      14,543,124 [17]       —N/a   
2       37785000       8231      4,591 [e]             37274000      13452   
3       32226000       2344     13,748 [f]             29000000       3483   
4       24073000       4333      5,556 [g]                 —N/a       —N/a   

                  
  Density (/km2)  
0     4,733 [14]  
1           —N/a  
2     2,771 [19]  
3     8,326 [21]  
4           —N/a

<h5 style="color:#1f77b4;">5. Renombrar columnas de forma explícita.</h5>

1. Recorre columna por columna
for col in df.columns
donde col es una tupla tipo ("Urban area[12]", "Population").

2. Convierte cada parte a texto y elimina las partes vacías (nan)
[str(x) for x in col if str(x) != "nan"]
A veces pandas pone nan cuando hay celdas de encabezado vacías en alguna fila.
Esto las saca para que no te quede “nan Population”.

3. Une las partes con un espacio y limpia bordes
" ".join(...).strip()

In [44]:
# 5) Aplanar columnas si vienen en MultiIndex
if isinstance(df.columns, pd.MultiIndex):   # Detecta si las columnas vienen “dobles”
    df.columns = [
        " ".join([str(x) for x in col if str(x) != "nan"]).strip() 
        for col in df.columns               # Recorre columna por columna
    ]

In [45]:
df.columns

Index(['City[a] City[a]', 'Country Country',
       'UN 2025 population estimates[11] UN 2025 population estimates[11]',
       'City proper[b] Definition', 'City proper[b] Population',
       'City proper[b] Area (km2)', 'City proper[b] Density (/km2)',
       'Urban area[12] Population', 'Urban area[12] Area (km2)',
       'Urban area[12] Density (/km2)', 'Metropolitan area[c] Population',
       'Metropolitan area[c] Area (km2)',
       'Metropolitan area[c] Density (/km2)'],
      dtype='object')

In [46]:
df.head()

,City[a] City[a],Country Country,UN 2025 population estimates[11] UN 2025 population estimates[11],City proper[b] Definition,City proper[b] Population,City proper[b] Area (km2),City proper[b] Density (/km2),Urban area[12] Population,Urban area[12] Area (km2),Urban area[12] Density (/km2),Metropolitan area[c] Population,Metropolitan area[c] Area (km2),Metropolitan area[c] Density (/km2)
0,Jakarta,Indonesia,41913860,Special region,10154134,664,"15,292 [13]",33756000,3546,"9,519 [d]",33430285,7063,"4,733 [14]"
1,Dhaka,Bangladesh,36585479,Capital city,10295407,338,"30,460 [15][16]",19134000,619,30911,"14,543,124 [17]",—N/a,—N/a
2,Tokyo,Japan,33412512,Metropolis prefecture,13515271,2191,"6,169 [18]",37785000,8231,"4,591 [e]",37274000,13452,"2,771 [19]"
3,Delhi,India,30222405,National Capital Territory,16753235,1484,"11,289 [20]",32226000,2344,"13,748 [f]",29000000,3483,"8,326 [21]"
4,Shanghai,China,29558908,Municipality,24870895,6341,"3,922 [22][23]",24073000,4333,"5,556 [g]",—N/a,—N/a,—N/a


In [52]:
# Renombrar explícitamente algunas claves (para que sea más legible)
df = df.rename(columns={
    #"City:prop_def": "City_prop_def"
    "City[a] City[a]": "City",
})

In [53]:
df.head()

,City,Country Country,UN 2025 population estimates[11] UN 2025 population estimates[11],City proper[b] Definition,City proper[b] Population,City proper[b] Area (km2),City proper[b] Density (/km2),Urban area[12] Population,Urban area[12] Area (km2),Urban area[12] Density (/km2),Metropolitan area[c] Population,Metropolitan area[c] Area (km2),Metropolitan area[c] Density (/km2)
0,Jakarta,Indonesia,41913860,Special region,10154134.0,664.0,15292.0,33756000.0,3546.0,9519.0,33430285.0,7063.0,4733.0
1,Dhaka,Bangladesh,36585479,Capital city,10295407.0,338.0,30460.0,19134000.0,619.0,30911.0,14543124.0,NaN,NaN
2,Tokyo,Japan,33412512,Metropolis prefecture,13515271.0,2191.0,6169.0,37785000.0,8231.0,4591.0,37274000.0,13452.0,2771.0
3,Delhi,India,30222405,National Capital Territory,16753235.0,1484.0,11289.0,32226000.0,2344.0,13748.0,29000000.0,3483.0,8326.0
4,Shanghai,China,29558908,Municipality,24870895.0,6341.0,3922.0,24073000.0,4333.0,5556.0,NaN,NaN,NaN


<h5 style="color:#1f77b4;">6. Limpiar datos</h5>

In [54]:
# 6) Limpieza y conversión de tipos
# - quitar referencias tipo [13] en toda la tabla
df = df.replace(r"\[.*?\]", "", regex=True)

In [55]:
df.head()

,City,Country Country,UN 2025 population estimates[11] UN 2025 population estimates[11],City proper[b] Definition,City proper[b] Population,City proper[b] Area (km2),City proper[b] Density (/km2),Urban area[12] Population,Urban area[12] Area (km2),Urban area[12] Density (/km2),Metropolitan area[c] Population,Metropolitan area[c] Area (km2),Metropolitan area[c] Density (/km2)
0,Jakarta,Indonesia,41913860,Special region,10154134.0,664.0,15292.0,33756000.0,3546.0,9519.0,33430285.0,7063.0,4733.0
1,Dhaka,Bangladesh,36585479,Capital city,10295407.0,338.0,30460.0,19134000.0,619.0,30911.0,14543124.0,NaN,NaN
2,Tokyo,Japan,33412512,Metropolis prefecture,13515271.0,2191.0,6169.0,37785000.0,8231.0,4591.0,37274000.0,13452.0,2771.0
3,Delhi,India,30222405,National Capital Territory,16753235.0,1484.0,11289.0,32226000.0,2344.0,13748.0,29000000.0,3483.0,8326.0
4,Shanghai,China,29558908,Municipality,24870895.0,6341.0,3922.0,24073000.0,4333.0,5556.0,NaN,NaN,NaN


In [56]:
# - convertir columnas numéricas (población/área/densidad) eliminando comas y guiones raros
def to_number(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", "", regex=False)
         .str.replace("—", "", regex=False)
         .str.replace("–", "", regex=False)
         .str.strip(),
        errors="coerce"
    )

for c in df.columns:
    if any(k in c.lower() for k in ["pop", "population", "area", "density"]):
        df[c] = to_number(df[c])

# - eliminar filas “vacías” (por ejemplo, donde no hay ciudad)
df = df.dropna(subset=["City"])

In [57]:
df.head()

,City,Country Country,UN 2025 population estimates[11] UN 2025 population estimates[11],City proper[b] Definition,City proper[b] Population,City proper[b] Area (km2),City proper[b] Density (/km2),Urban area[12] Population,Urban area[12] Area (km2),Urban area[12] Density (/km2),Metropolitan area[c] Population,Metropolitan area[c] Area (km2),Metropolitan area[c] Density (/km2)
0,Jakarta,Indonesia,41913860,Special region,10154134.0,664.0,15292.0,33756000.0,3546.0,9519.0,33430285.0,7063.0,4733.0
1,Dhaka,Bangladesh,36585479,Capital city,10295407.0,338.0,30460.0,19134000.0,619.0,30911.0,14543124.0,NaN,NaN
2,Tokyo,Japan,33412512,Metropolis prefecture,13515271.0,2191.0,6169.0,37785000.0,8231.0,4591.0,37274000.0,13452.0,2771.0
3,Delhi,India,30222405,National Capital Territory,16753235.0,1484.0,11289.0,32226000.0,2344.0,13748.0,29000000.0,3483.0,8326.0
4,Shanghai,China,29558908,Municipality,24870895.0,6341.0,3922.0,24073000.0,4333.0,5556.0,NaN,NaN,NaN


<h5 style="color:#1f77b4;">7. Guardar el resultado en un archivo CSV con to_csv().</h5>

In [58]:
df.to_csv("largest_cities_clean.csv", index=False, encoding="utf-8")


In [59]:
df2 = pd.read_csv("largest_cities_clean.csv")


In [61]:
df2.head(10)

,City,Country Country,UN 2025 population estimates[11] UN 2025 population estimates[11],City proper[b] Definition,City proper[b] Population,City proper[b] Area (km2),City proper[b] Density (/km2),Urban area[12] Population,Urban area[12] Area (km2),Urban area[12] Density (/km2),Metropolitan area[c] Population,Metropolitan area[c] Area (km2),Metropolitan area[c] Density (/km2)
0,Jakarta,Indonesia,41913860,Special region,10154134.0,664.0,15292.0,33756000.0,3546.0,9519.0,33430285.0,7063.0,4733.0
1,Dhaka,Bangladesh,36585479,Capital city,10295407.0,338.0,30460.0,19134000.0,619.0,30911.0,14543124.0,NaN,NaN
2,Tokyo,Japan,33412512,Metropolis prefecture,13515271.0,2191.0,6169.0,37785000.0,8231.0,4591.0,37274000.0,13452.0,2771.0
3,Delhi,India,30222405,National Capital Territory,16753235.0,1484.0,11289.0,32226000.0,2344.0,13748.0,29000000.0,3483.0,8326.0
4,Shanghai,China,29558908,Municipality,24870895.0,6341.0,3922.0,24073000.0,4333.0,5556.0,NaN,NaN,NaN
5,Guangzhou,China,27563372,City (sub-provincial),18676605.0,7434.0,2512.0,26940000.0,4535.0,5940.0,NaN,NaN,NaN
6,Cairo,Egypt,25566102,Urban governorate,10044894.0,3085.0,3256.0,20296000.0,2010.0,10098.0,NaN,NaN,NaN
7,Manila,Philippines,24735305,Capital city,1780148.0,43.0,41399.0,24922000.0,1911.0,13041.0,12877253.0,620.0,20770.0
8,Kolkata,India,22549738,Municipal corporation,4496694.0,206.0,21829.0,21747000.0,1352.0,16085.0,15333000.0,1887.0,8126.0
9,Seoul,South Korea,22490482,Special city,10013781.0,605.0,16552.0,23016000.0,2769.0,8312.0,25514000.0,11704.0,2180.0
